# CSV Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Build a gradebook.** CSV is plain text with commas — a triple-quoted string plus `write_text` builds one from scratch.

In [ ]:
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)

raw_csv = """Name,Math,Science
Sarah,88,92
Tanvir,75,81
Amina,95,89
"""

class_path = Path("sample_data", "class.csv")
class_path.write_text(raw_csv, encoding="utf-8")

print(class_path.read_text(encoding="utf-8"))

**2. Rows come back as lists.** `csv.reader` yields one list of strings per row — even the numbers arrive as text.

In [ ]:
import csv

with open("sample_data/class.csv", newline="", encoding="utf-8") as f:
    for row in csv.reader(f):
        print(row, "<-", type(row).__name__)
# ['Name', 'Math', 'Science'] <- list
# ['Sarah', '88', '92'] <- list
# ['Tanvir', '75', '81'] <- list
# ['Amina', '95', '89'] <- list

**3. Peel off the header.** `next(reader)` consumes the header so the remaining iterations are pure data.

In [ ]:
import csv

with open("sample_data/class.csv", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)            # pull off the header...
    first_student = next(reader)     # ...then the data starts clean

print("Header:", header)
print("Columns:", len(header))
print("First student:", first_student)

## Part 2 — Practice

**4. Write it properly.** `writerow()` emits the header, `writerows()` emits every data row, and quoting is handled for you.

In [ ]:
import csv
from pathlib import Path

rows = [
    ["Subject", "Credits", "Grade"],
    ["Math", 4, "A"],
    ["Science", 3, "A-"],
    ["History", 2, "B+"],
]

with open("sample_data/subjects.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(rows[0])         # header, one row
    writer.writerows(rows[1:])       # data, many rows at once

print(Path("sample_data/subjects.csv").read_text(encoding="utf-8"))

**5. Commas inside values.** The writer quoted `India, West Bengal` automatically — a hand-built row would have split that value in two.

In [ ]:
import csv
from pathlib import Path

cities = [
    ["City", "Country"],
    ["Chattogram", "Bangladesh"],
    ["Kolkata", "India, West Bengal"],
]

with open("sample_data/cities.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(cities)

print(Path("sample_data/cities.csv").read_text(encoding="utf-8"))
# "India, West Bengal" lands QUOTED: the comma stays inside one field.
# ",".join(...) would have split it into two fake columns.

**6. DictReader by name.** Rows become dictionaries keyed by the header, so values are fetched by meaning — converted with `int()` because they arrive as strings.

In [ ]:
import csv
from statistics import mean

math_marks = []
with open("sample_data/class.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        math_marks.append(int(row["Math"]))   # strings in, ints out
        print(row["Name"], "->", row["Math"])

print("Class Math average:", round(mean(math_marks), 1))
# Sarah -> 88 / Tanvir -> 75 / Amina -> 95 / Class Math average: 86.0

**7. Why `newline=""` matters.** Omitting the flag doubles the carriage returns on Windows (`\\r\\r\\n`), which Excel renders as blank rows between records.

In [ ]:
import csv
from pathlib import Path

rows = [["City", "Population"], ["Dhaka", 22479000]]

with open("sample_data/no_flag.csv", "w", encoding="utf-8") as f:       # no flag
    csv.writer(f).writerows(rows)

with open("sample_data/with_flag.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(rows)

print("without flag:", repr(Path("sample_data/no_flag.csv").read_bytes()))
print("with flag   :", repr(Path("sample_data/with_flag.csv").read_bytes()))
# Without the flag Windows turns each \r\n into \r\r\n - Excel shows a
# blank row between every record. newline="" lets csv control endings.

## Part 3 — Challenge

**8. Export averages with `DictWriter`.** Declare `fieldnames` up front, `writeheader()`, then `writerows()` — dicts go out as ordered columns.

In [ ]:
import csv
from statistics import mean
from pathlib import Path

summary = []
with open("sample_data/class.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        marks = [int(row["Math"]), int(row["Science"])]
        summary.append({"Student": row["Name"], "Average": round(mean(marks), 1)})

with open("sample_data/student_averages.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["Student", "Average"])
    writer.writeheader()
    writer.writerows(summary)

print(Path("sample_data/student_averages.csv").read_text(encoding="utf-8"))
# Student,Average / Sarah,90.0 / Tanvir,78.0 / Amina,92.0

**9. Delimiter detective.** Any `delimiter` works for writing — and must be repeated for reading; European exports favour `;`.

In [ ]:
import csv
from pathlib import Path

scores = [["name", "score"], ["Sarah", 88], ["Tanvir", 75]]

with open("sample_data/scores.tsv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f, delimiter="\t").writerows(scores)

with open("sample_data/scores_semicolon.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f, delimiter=";").writerows(scores)

print("tsv       :", repr(Path("sample_data/scores.tsv").read_text(encoding="utf-8")))
print("semicolon :", repr(Path("sample_data/scores_semicolon.csv").read_text(encoding="utf-8")))

with open("sample_data/scores_semicolon.csv", newline="", encoding="utf-8") as f:
    for row in csv.reader(f, delimiter=";"):
        print(row)

# European locales often export with ";" because their comma is the
# decimal mark.